# Development and validation sampling

Use an out-of-time split to preserve temporal ordering and provide a realistic validation sample.

## Selected sample split

| Dataset | Application dates | Accounts | Known Good | Known Bad | Bad rate |
|---|---|---:|---:|---:|---:|
| Build | 1 Jan 2023–30 Jun 2024 | 7,258 | 6,964 | 294 | **4.05%** |
| Out-of-sample | 1 Jul–30 Dec 2024 | 2,458 | 2,356 | 102 | **4.15%** |
| Total | Full period | 9,716 | 9,320 | 396 | **4.08%** |

This produces a conventional 75/25 split. An 80/20 split contained only 80 bad accounts in the validation sample, which was too small for reliable evaluation. The later period also has a slightly higher observed bad rate, making it a useful test of temporal stability.

---

In [1]:
# Locating and opening the KGB dataset

from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the data folder from the project root or notebooks folder.
current_folder = Path.cwd()

if (current_folder / "data" / "KGB model.csv").is_file():
    DATA_DIR = current_folder / "data"
elif (current_folder.parent / "data" / "KGB model.csv").is_file():
    DATA_DIR = current_folder.parent / "data"
else:
    DATA_DIR = Path(r"C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data")

input_path = DATA_DIR / "KGB model.csv"

if not input_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {input_path}")

working_data = pd.read_csv(input_path)

print(f"Loaded {len(working_data):,} accounts from {input_path}")

Loaded 9,716 accounts from c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB model.csv


In [2]:
# Building the sample split

# Split the retained KGB population by application date.
model_data = working_data.copy()

model_data["application_date"] = pd.to_datetime(
    model_data["application_date"],
    errors="raise",
)

if model_data["application_date"].isna().any():
    raise ValueError("Missing application dates must be resolved before splitting.")

cutoff_date = pd.Timestamp("2024-07-01")

build_data = model_data.loc[model_data["application_date"] < cutoff_date].copy()

out_sample_data = model_data.loc[model_data["application_date"] >= cutoff_date].copy()

# Review counts and bad rates.
split_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "accounts": len(data),
            "known_good": data["target_bad"].eq(0).sum(),
            "known_bad": data["target_bad"].eq(1).sum(),
            "bad_rate_pct": data["target_bad"].mean() * 100,
        }
        for name, data in [
            ("Build", build_data),
            ("Out-of-sample", out_sample_data),
        ]
    ]
)

display(split_summary.style.format({"bad_rate_pct": "{:.2f}%"}))

,dataset,accounts,known_good,known_bad,bad_rate_pct
0,Build,7258,6964,294,4.05%
1,Out-of-sample,2458,2356,102,4.15%


In [3]:
# Creating and exporting the new datasets

# Export the build (in-sample) and out-of-sample datasets.
in_sample_path = DATA_DIR / "KGB in sample.csv"
out_sample_path = DATA_DIR / "KGB out sample.csv"

build_data.to_csv(in_sample_path, index=False)
out_sample_data.to_csv(out_sample_path, index=False)

print(f"In-sample: {len(build_data):,} accounts → {in_sample_path}")
print(f"Out-of-sample: {len(out_sample_data):,} accounts → {out_sample_path}")

In-sample: 7,258 accounts → c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB in sample.csv
Out-of-sample: 2,458 accounts → c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB out sample.csv
